<a href="https://colab.research.google.com/github/ksuaray/M4DS/blob/MATH-170-Spring-2026/Lab7_Riemann_Sums_Definite_Integrals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**MATH 170: Calculus for Data Science**

# **Lab 2 (AN): Riemann Sums, Definite Integrals, and a First Look at Probability Density**



**Theme:** *Approximation* (A) + *Visualization* (V) + *Aggregation* (A)

### What you'll practice

1. **Riemann sums:** compute left, right, and midpoint approximations.
2. **Visualization:** use **Plotly** to see rectangles and how they change as $n$ changes.
3. **Convergence:** compare approximations to the exact definite integral.
4. **Soft intro to density:** reinterpret area as “how much” and begin thinking of area as probability.

---

### Main functions from class

We will use the same functions from AN2:

- $f(x) = x^2$ on $[0,6]$
- $g(x) = e^{-x^2} - 0.5$ on $[-3,4]$

The big question of the lab is:

> How do $S_L(n)$, $S_R(n)$, and $S_M(n)$ behave, and what happens as $n$ gets large?


## 0. Setup
Run the next cell first.

In [ ]:

import numpy as np
import pandas as pd
import sympy as sp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, Markdown

sp.init_printing()
x = sp.Symbol('x', real=True)

print('Ready!')



## 0.1 Helper functions

These helpers will do the repetitive work for us so we can focus on the ideas.


In [ ]:

def sample_points(a, b, n, method='L'):
    dx = (b - a) / n
    if method == 'L':
        xs = a + np.arange(n) * dx
    elif method == 'R':
        xs = a + np.arange(1, n + 1) * dx
    elif method == 'M':
        xs = a + (np.arange(n) + 0.5) * dx
    else:
        raise ValueError("method must be 'L', 'R', or 'M'")
    return xs, dx


def riemann_sum(f_num, a, b, n, method='L'):
    xs, dx = sample_points(a, b, n, method)
    return np.sum(f_num(xs)) * dx


def riemann_table(f_num, a, b, n, method='L'):
    xs, dx = sample_points(a, b, n, method)
    vals = f_num(xs)
    left_edges = a + np.arange(n) * dx
    right_edges = a + np.arange(1, n + 1) * dx
    return pd.DataFrame({
        'subinterval': [f'[{left_edges[i]:.3g}, {right_edges[i]:.3g}]' for i in range(n)],
        'sample_x': xs,
        'height': vals,
        'rect_area': vals * dx
    })


def exact_integral(expr, a, b):
    return sp.integrate(expr, (x, a, b))


def plot_riemann(f_num, a, b, n=4, method='L', title='Riemann Sum'):
    xs_curve = np.linspace(a, b, 600)
    ys_curve = f_num(xs_curve)
    xs, dx = sample_points(a, b, n, method)
    heights = f_num(xs)
    left_edges = a + np.arange(n) * dx

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=xs_curve, y=ys_curve,
        mode='lines', name='curve',
        line=dict(width=3)
    ))

    for i in range(n):
        x0 = left_edges[i]
        x1 = x0 + dx
        h = float(heights[i])
        fig.add_shape(
            type='rect',
            x0=x0, x1=x1, y0=0, y1=h,
            line=dict(width=1),
            fillcolor='rgba(99, 110, 250, 0.28)'
        )
        fig.add_trace(go.Scatter(
            x=[xs[i]], y=[h],
            mode='markers',
            marker=dict(size=8),
            name='sample point' if i == 0 else None,
            showlegend=(i == 0)
        ))

    fig.add_hline(y=0, line_width=1)
    fig.update_layout(
        title=f'{title}: method={method}, n={n}',
        xaxis_title='x',
        yaxis_title='y',
        template='plotly_white',
        width=850,
        height=450
    )
    return fig


def convergence_table(f_num, expr, a, b, n_values, methods=('L','R','M')):
    exact_val = float(sp.N(exact_integral(expr, a, b)))
    rows = []
    for n in n_values:
        row = {'n': n, 'exact': exact_val}
        for method in methods:
            approx = riemann_sum(f_num, a, b, n, method)
            row[f'{method}_approx'] = approx
            row[f'{method}_error'] = approx - exact_val
        rows.append(row)
    return pd.DataFrame(rows)



---
# Part 1 — The classic example from AN2: $f(x) = x^2$ on $[0,6]$

We start with the same function from class. This is a good first test because:

- the graph is easy to understand,
- left/right/midpoint sums behave differently,
- and we can compute the exact integral.



## 1.1 Plot the curve

**GTW:** Run the next cell. Then write 1–2 sentences answering:

- Is $f(x)=x^2$ increasing or decreasing on $[0,6]$?
- Based on the picture alone, which do you think is bigger: $S_L(n)$ or $S_R(n)$?


In [ ]:

f_expr = x**2
f_num = sp.lambdify(x, f_expr, 'numpy')

xs = np.linspace(0, 6, 400)
fig = go.Figure()
fig.add_trace(go.Scatter(x=xs, y=f_num(xs), mode='lines', name='x^2', line=dict(width=3)))
fig.add_hline(y=0, line_width=1)
fig.update_layout(title='f(x) = x^2 on [0,6]', template='plotly_white', width=850, height=420)
fig.show()



## 1.2 Compute $S_L(3)$, $S_R(3)$, and $S_M(3)$

First we'll visualize the graph with the rectangles, then let's compute using each of the methods.


Graph:

In [ ]:
fig = plot_riemann(f_num, 0, 6, n=3, method='L', title='Riemann rectangles for x^2')
fig.show()

## **GTW:** In the graph above, switch between the L, R and M rules to visualize the example from class.

Compute:

In [ ]:

for method in ['L', 'R', 'M']:
    print(f"{method}3 =", riemann_sum(f_num, 0, 6, 3, method))


This matches the first approximations from the AN2 class notes.


## **GTW:** Use the next cell to inspect the rectangles numerically. What do these numbers mean?


In [ ]:

riemann_table(f_num, 0, 6, 3, 'M')



## 1.3 Visualizing Limits: switch method and increase $n$

Use the the following to compare:

- **L** = left endpoint,
- **R** = right endpoint,
- **M** = midpoint.

## **GTW:** In the cell below, create a variable called `height` that will allow you to select the L, R or M method, and a variable called `rect` that will allow you to choose the number of rectangles. Place them in the appropriate spot in the `plot_riemann` function.

In [ ]:
        #Define the height variable here
        #Define the rect variable here

fig = plot_riemann(f_num, 0, 6, n=3, method='L', title='Riemann rectangles for x^2')
fig.show()


## 1.4 Exact definite integral

Now compute the **exact** value using SymPy:
$$
\text{Exact Area} = \int_0^6 x^2 dx
$$


In [ ]:

exact_f = exact_integral(f_expr, 0, 6)
exact_f



### Think

1. Which method seems to **underestimate** for $x^2$ on $[0,6]$?
2. Which method seems to **overestimate**?
3. Why does midpoint often do surprisingly well?



## 1.5 Convergence table

Let’s compare errors as $n$ increases.


In [ ]:

conv_f = convergence_table(f_num, f_expr, 0, 6, [3, 6, 12, 24, 48])
conv_f.round(6)



## **GTW:** In the cell below, describe what happens to the errors as $n$ grows.



---
# Part 2 — A more interesting function: $g(x) = e^{-x^2} - 0.5$ on $[-3,4]$

This is the second function from AN2. It is more interesting because the graph is sometimes above the $x$-axis and sometimes below it.

That means the definite integral is **signed area**.



## 2.1 Plot the curve

## **GTW:** Before running the integral, make a prediction:

- Will the total signed area be positive, negative, or close to zero?


In [ ]:

g_expr = sp.exp(-x**2) - sp.Rational(1,2)
g_num = sp.lambdify(x, g_expr, 'numpy')

xs = np.linspace(-3, 4, 500)
fig = go.Figure()
fig.add_trace(go.Scatter(x=xs, y=g_num(xs), mode='lines', name='$e^{-x^2} - 0.5$', line=dict(width=3)))
fig.add_hline(y=0, line_width=1)
fig.update_layout(title='$g(x) = e^{-x^2} - 0.5$', template='plotly_white', width=850, height=420)
fig.show()



## 2.2 Visualize and Compute the worksheet approximation $S_L(7)$


Using the code from above, provide the code to visualize $S_L(n)$, $S_R(n)$ and $S_M(n)$ for $n=7,14,28$ and $56$.

In [ ]:
...
...

fig = plot_riemann(g_num, -3, 4, n=..., method=..., title='Riemann rectangles for x^2')
fig.show()

In [ ]:

riemann_sum(g_num, -3, 4, 7, 'L')



## 2.3 Interactive Plotly slider

Try different methods and values of $n$.

Pay attention to rectangles that drop **below** the axis.



## 2.3 Exact integral and comparison


In [ ]:

exact_g = exact_integral(g_expr, -3, 4)
exact_g, float(sp.N(exact_g))


## **GTW:** Fill in the ... below based on your experiments from above

In [ ]:

conv_g = convergence_table(g_num, g_expr, -3, 4, [...])
conv_g.round(6)



### Think

- Why is this example different from $x^2$?
- What do negative rectangles mean in context?
- Does “more rectangles” still help? Explain briefly.




So far, area has meant “how much is under the graph.”

A better phrase is:

> A definite integral measures **accumulated contribution**.

If the function is positive, it adds to the total.
If the function is negative, it subtracts from the total.



---
# Part 4 — A very soft intro to probability density

Here is the new idea:

If a curve is always nonnegative and its total area is $1$, we can interpret it as a **probability density**.

Then:

- area over an interval = probability of landing in that interval.

We are **not** doing a full probability unit yet. This is just a first look.


Let's motivate this example with **real data from Kaggle**. Check out a description [here](https://www.kaggle.com/datasets/valakhorasani/gym-members-exercise-dataset).

In [ ]:
# Install kagglehub for easy Kaggle data download
!pip install kagglehub --upgrade --quiet

import kagglehub
import pandas as pd
import plotly.express as px
import os

# Download the dataset using kagglehub
print("Downloading 'gym-members-exercise-dataset' from KaggleHub...")
path = kagglehub.dataset_download("valakhorasani/gym-members-exercise-dataset")
print(f"Dataset downloaded to: {path}")

# List contents of the downloaded directory to find the correct CSV file
print("Contents of downloaded directory:")
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            csv_file_path = os.path.join(root, file)
            print(f"Found CSV file: {csv_file_path}")
            break
    if 'csv_file_path' in locals():
        break

if 'csv_file_path' not in locals():
    raise FileNotFoundError("No CSV file found in the downloaded dataset.")

# Load the dataset
df = pd.read_csv(csv_file_path)


In [ ]:
df

Let's visualize a **histogram** of the `Calories_Burned` variable for male gym members:

In [ ]:
df1 = df[df['Gender']=='Male']
# Use `variable` for the bell-shaped distribution example

variable = 'Calories_Burned'
selected_column = df1[variable]

# Create a histogram using Plotly Express
fig = px.histogram(selected_column, nbins=13, title=f'Distribution of {variable}', histnorm='probability density')
fig.update_layout(xaxis_title= f'{variable}', yaxis_title='Density', template='plotly_white')
fig.show()

xbar = selected_column.mean()
sdx = selected_column.std()

print(f"Mean {variable}: {xbar:.2f} units")
print(f"Standard Deviation of {variable}: {sdx:.2f} units")

The **bell shape** is no coincidence. It reflects the **normal distribution**, an underlying probabilistic structure that shows up in social , life and physical sciences, among many other places. The mathematical function that describes it is discussed below.


## 4.1 Start with a bell-shaped curve

We’ll use $p(x)$, which will be a shifted and scaled version of the *standard normal probability density function*

$$
f(x) = \frac{1}{\sqrt{\pi}} e^{-x^2}.
$$

This scaling makes the total area over all real numbers equal to $1$.

Let's see what it looks like on the histogram below:


In [ ]:
import numpy as np
import pandas as pd
import sympy as sp
import plotly.graph_objects as go
import plotly.express as px

# Re-initialize sympy symbol for p_expr
x = sp.Symbol('x', real=True)

# --- Recreate histogram (fig) related variables ---
df1 = df[df['Gender']=='Male']
variable = 'Calories_Burned'
selected_column = df1[variable]

# Create the histogram figure
fig = px.histogram(selected_column, nbins=13, title=f'Distribution of {variable} with Density Curve', histnorm='probability density')
fig.update_layout(xaxis_title=f'{variable}', yaxis_title='Density', template='plotly_white')

# --- Recreate bell-shaped curve (fig1) related variables ---
xbar = selected_column.mean()
sdx = selected_column.std()
p_expr = sp.exp(-((x-xbar)/sdx)**2) / (sp.sqrt(sp.pi)*sdx)
p_num = sp.lambdify(x, p_expr, 'numpy')
xs_curve = np.linspace(selected_column.min(), selected_column.max(), 600)

# Add the density curve as a scatter trace to the histogram figure
fig.add_trace(go.Scatter(x=xs_curve, y=p_num(xs_curve),
                         mode='lines', name='p(x) (Normal PDF)',
                         line=dict(color='red', width=3)))

fig.show()


## 4.2 Approximate total area on a large window

We cannot capture all real numbers in a notebook plot, so we use a large interval like $[84.7,1804.2]$.

If the area is close to $1$, that supports the density interpretation.


In [ ]:
a = -3*sdx+xbar
b = 3*sdx+xbar
for n in [20, 100, 400]:
    print(f'M{n} on [{a},{b}] =', midpoint_sum := riemann_sum(p_num, -3*sdx+xbar, 3*sdx+xbar, n, 'M'))



## 4.3 Approximate a probability

Now interpret area on an interval as probability.

For example going back to the original $p(x)$, the probability it falls between $a$ and $b$ is

$$
P(a \le X \le b) \approx \int_{a}^{b} p(x)\,dx.
$$


In [ ]:
c = 800
d = 1100

prob_approx = riemann_sum(p_num, c, d, 200, 'M')
prob_exact = exact_integral(p_expr, c, d)

fig = go.Figure()
fig.add_trace(go.Scatter(x=xs_curve, y=p_num(xs_curve),
                         mode='lines', name='p(x) (Normal PDF)',
                         line=dict(color='red', width=3)))

# Shade the area from c to d
xs_shade = np.linspace(c, d, 100)
ys_shade = p_num(xs_shade)
fig.add_trace(go.Scatter(
    x=np.concatenate(([c], xs_shade, [d])),
    y=np.concatenate(([0], ys_shade, [0])),
    fill='toself', mode='lines', line=dict(width=0),
    fillcolor='rgba(0, 100, 80, 0.2)', name=f'Area from {c} to {d}'
))

fig.update_layout(title='p_num Density with Shaded Area',
                  xaxis_title='x', yaxis_title='Density',
                  template='plotly_white', width=850, height=420)
fig.show()
print('Approximate probability on [c,d] =', prob_approx)
print('Exact value from SymPy =', prob_exact)
print('Decimal ≈', float(sp.N(prob_exact)))



# **LAB 7 Homework**

Complete the following activities by 11:59pm on Wednesday 4/22. Submit the url (link) to this notebook in Canvas once you complete these exercises. You may work collaboratively with your classmates, but each student will be expected to submit their own work.


Write a short response to each:

1. For an increasing function like $x^2$, how do $S_L(n)$ and $S_R(n)$ compare to the exact integral? In other words, if the value of the exact integral is $I$, and the function is increasing, write an **inequality** that relates the three quantities.


2. Why does increasing $n$ improve Riemann sum approximations?



3. What is the difference between ordinary area and **signed** area?


4. In your own words, what does it mean for area to become probability?

5. Use the code above to complete 5.3.33, 34 and 40 from the [textbook](https://opentext.uleth.ca/apex-standard/sec_riemann.html). Place your code and work below:

6. Find the probability that a randomly selected male gym member burnt between 1000 and 1500 calories. Is it more or less than the probability they burnt between 500 and 1000? Why do you think that is the case?